In [1]:
import os
import random
import pandas as pd
import numpy as np
from pathlib import Path
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from PatchDataset import load_dataset

PATCHES_ROOT = "patches_dataset/patches_v3_seed42_pad1.6_iouth0.09"
csv_path = os.path.join(PATCHES_ROOT, "metadata.csv")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PATCH_SIZE = 224
BATCH_SIZE = 32
AUG_MULTIPLIER = 2.0  # x2 training size
TARGET_POS_RATIO = 0.60


In [2]:
# loading the dataset
df = pd.read_csv(csv_path)
df["patch_id"] = df["patch_id"].astype(int)
df["label"] = df["label"].astype(int)
df["type"] = df["type"].astype(str)

train_dataset,test_dataset,train_loader,test_loader = load_dataset(df,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

train/test: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0.) tensor(1.)


In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print("Train balance:", train_df["label"].value_counts(normalize=True))
print("Test balance:", test_df["label"].value_counts(normalize=True))

Train: 15858 | Test: 3965
Train balance: label
1    0.684891
0    0.315109
Name: proportion, dtype: float64
Test balance: label
1    0.684994
0    0.315006
Name: proportion, dtype: float64


In [4]:
AUG_ROOT = "patches_dataset/patches_v3_train_aug"
os.makedirs(os.path.join(AUG_ROOT, "positive"), exist_ok=True)
os.makedirs(os.path.join(AUG_ROOT, "negative"), exist_ok=True)

print("Augmented directories created")

Augmented directories created


In [5]:
train_aug_pos = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.08, rotate_limit=20, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
])

train_aug_neg = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.06, rotate_limit=15, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
])

eval_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

/home/alant/myglobal/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
def generate_augmented_samples(df_train, aug_root, multiplier=2.0, pos_ratio=0.60):
    train_pos = df_train[df_train["label"] == 1].copy()
    train_neg = df_train[df_train["label"] == 0].copy()
    
    P, N = len(train_pos), len(train_neg)
    T = int(multiplier * (P + N))
    target_pos, target_neg = int(pos_ratio * T), T - int(pos_ratio * T)
    
    aug_pos_needed = max(0, target_pos - P)
    aug_neg_needed = max(0, target_neg - N)
    
    print(f"Original train: {P+N} (P:{P}, N:{N})")
    print(f"Target: {T} (P:{target_pos}, N:{target_neg})")
    print(f"Need to augment: pos={aug_pos_needed}, neg={aug_neg_needed}")
    
    aug_metadata = []
    
    # Augment positives
    pos_dir_orig = os.path.join(PATCHES_ROOT, "positive")
    pos_dir_aug = os.path.join(aug_root, "positive")
    
    for i in range(aug_pos_needed):
        orig_idx = i % len(train_pos) # so that it doesnt go out of bounds
        orig_patch_id = train_pos.iloc[orig_idx]["patch_id"]
        
        img_path = os.path.join(pos_dir_orig, f"{orig_patch_id}.png")
        img = cv2.imread(img_path)
        if img is None: 
            print(f"Warning: Could not read image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        aug_img = train_aug_pos(image=img)["image"]
        new_pid = f"patch{orig_patch_id}_aug_{i}"
        cv2.imwrite(os.path.join(pos_dir_aug, f"{new_pid}.png"), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
        
        aug_metadata.append({
            "patch_id": new_pid,
            "label": 1,
            "type": "positive"
        })
    
    # Augment negatives
    neg_dir_orig = os.path.join(PATCHES_ROOT, "negative")
    neg_dir_aug = os.path.join(aug_root, "negative")
    
    for i in range(aug_neg_needed):
        orig_idx = i % len(train_neg)
        orig_patch_id = train_neg.iloc[orig_idx]["patch_id"]
        
        img_path = os.path.join(neg_dir_orig, f"{orig_patch_id}.png")
        img = cv2.imread(img_path)
        if img is None: 
            print(f"Warning: Could not read image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        aug_img = train_aug_neg(image=img)["image"]
        new_pid = f"patch{orig_patch_id}_aug_{i}"
        cv2.imwrite(os.path.join(neg_dir_aug, f"{new_pid}.png"), 
                   cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
        
        aug_metadata.append({
            "patch_id": new_pid,
            "label": 0,
            "type": "negative"
        })
    
    aug_df = pd.DataFrame(aug_metadata)
    return aug_df


Original train: 15858 (P:10861, N:4997)
Target: 31716 (P:19029, N:12687)
Need to augment: pos=8168, neg=7690
Generated 15858 augmented samples


In [ ]:
# Generate augmented training data
train_aug_df = generate_augmented_samples(train_df, AUG_ROOT)
print(f"Generated {len(train_aug_df)} augmented samples")

In [9]:
# dropping the columns cx and cy and filename
train_df_copy = train_df.copy().drop(columns=["cx", "cy", "filename"])
train_df_combined = pd.concat([
    train_df_copy,
    train_aug_df
], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Combined training dataset:")
print(train_df_combined["label"].value_counts())
print(train_df_combined["label"].value_counts(normalize=True))
df_combined_path = os.path.join(AUG_ROOT, "train_aug_metadata.csv")
train_df_combined.to_csv(df_combined_path, index=False)

Combined training dataset:
label
1    19029
0    12687
Name: count, dtype: int64
label
1    0.599981
0    0.400019
Name: proportion, dtype: float64


In [10]:
test_path = "patches_dataset/test_patches_v3/test_metadata.csv"
test_df.to_csv(test_path, index=False)
print("Test dataset (original, unchanged):")
print(test_df["label"].value_counts())

Test dataset (original, unchanged):
label
1    2716
0    1249
Name: count, dtype: int64
